# Document summary agent

Regulators publish long documents, and compliance teams have to turn each one into something a decision-maker
can act on: what the firm must do, by when, and what it means for *this* firm. In this workshop you'll build a
**deep agent** that does this for a policy statement, using two agents:

1. A **main agent** (the orchestrator) loads the document, which arrives already split into sections, and looks up
   the firm's profile.
2. It hands each section to a **sub-agent**, `section-summariser`, which summarises that one section and extracts
   its obligations as a structured `SectionSummary`. The sub-agents run in parallel.
3. The main agent collects the summaries and writes a brief for the firm.

The regulator, the firm and the document are fictional, and stored in `data/`. The only network call is to the
model.

| Path | What's in it |
|------|--------------|
| `agent.py` | The main agent, exported as `agent`. |
| `instructions.md` | The main agent's system prompt. |
| `sub_agent.py` | The `section-summariser` sub-agent and the `SectionSummary` it returns. |
| `sub_agent_instructions.md` | The sub-agent's system prompt. |
| `tools/` | The lookups the agent uses: documents, firm profiles and providers. |
| `data/` | The firm's database: one JSONL file per table, plus the documents themselves. |

**How to use this notebook:** run each cell in order with Shift+Enter.

## Step 0: Setup

Copy `.env.example` to `.env` in the repo root (shared by all three topics) and fill in your gateway key. To use
another model, set `MODEL_NAME` as `provider:model` (for example `openai:gpt-4.1-mini`). The model is set up once,
in `utils/models.py`, so you don't need to edit any code to switch.

In [1]:
# Notebook only: let Python find `agent.py`, `sub_agent.py` and the `utils/` and `tools/` folders.
# This notebook lives in modules/, so the workshop root is one level up.
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
from utils.models import get_model

print(get_model("orchestrator").invoke("Say hello to the compliance hackathon in one sentence.").text)

Hello, compliance hackathon: here's to building tools that make doing the right thing faster, easier, and a little more fun!


## Step 1: The tools

The data lives in `data/` as **JSONL** files: one file per table, one JSON object per line. `utils/db.py` reads
them. Two tools in `tools/` use them, and both return a JSON object, so the model gets clean, structured data:

- `get_document` returns the document's `document_id`, `title` and `sections`. Each section has its `number`,
  `title` and full `text`. We split at the `## ` headings, rather than every N characters, so each piece keeps its
  section number. We need those numbers later, to check every claim against its source.
- `get_firm_profile` returns the firm and its `third_party_providers`, each with its `contract_end` and whether it
  is `material`. The brief is for Harbour Mutual Bank, `FIRM-001`.

The shapes are defined as Pydantic models (`Document` and `Firm`) in those files.

In [3]:
from tools.documents import get_document
from tools.firms import get_firm_profile

document = get_document("PS26-4")
print(document["title"], "\n")
for section in document["sections"]:
    print(f"{section['number']:>2}. {section['title']:<55} {len(section['text'].split()):>4} words")

Policy Statement PS26/4: Operational resilience and third-party technology risk 

 1. Introduction and purpose                                 225 words
 2. Scope and application                                    180 words
 3. Definitions                                              159 words
 4. Important business services and impact tolerances        179 words
 5. Mapping and scenario testing                             165 words
 6. Third-party technology providers                         197 words
 7. Incident reporting                                       157 words
 8. Governance and accountability                            125 words
 9. Transitional arrangements                                123 words
10. Next steps and contacts                                   67 words


In [4]:
import json

print(json.dumps(get_firm_profile("FIRM-001"), indent=1))

{
 "firm_id": "FIRM-001",
 "name": "Harbour Mutual Bank",
 "type": "retail bank",
 "customers": 1200000,
 "total_assets": "\u00a39 billion",
 "operational_resilience_senior_manager": null,
 "notes": "Mobile banking and card payments both run on NimbusCloud.",
 "third_party_providers": [
  {
   "provider_id": "PROVIDER-001",
   "name": "NimbusCloud",
   "service": "core banking hosting",
   "contract_end": "30 June 2027",
   "material": true
  },
  {
   "provider_id": "PROVIDER-002",
   "name": "PayCore Ltd",
   "service": "card processing",
   "contract_end": "31 December 2029",
   "material": true
  }
 ]
}


## Step 2: The sub-agent

The sub-agent only ever sees one section, so it can focus on it. It is defined in `sub_agent.py` with three things:

- **Instructions** (`sub_agent_instructions.md`): what counts as an obligation, and the rules for accuracy.
- **A response format**: the `SectionSummary` Pydantic model. The model must fill it in, so every section comes
  back in the same shape.
- **A name and description**, which the main agent uses to decide when to call it.

The most important part of `SectionSummary` is the list of **obligations**: things the firm *must* do. For each
one, the model gives the paragraph number, the deadline exactly as written, and a quote copied word for word.
Those fields make the summary *reviewable*: a reviewer, or our code, can check each claim against the source.

In [5]:
from sub_agent import INSTRUCTIONS as SUB_AGENT_INSTRUCTIONS, SectionSummary

print(SUB_AGENT_INSTRUCTIONS)

# Section summariser agent
You are given one section: its number, title and text. Return a `SectionSummary` as a single JSON object, with no text before or after it:

- `number`: the section number, for example `"4"`.
- `title`: the section title.
- `summary`: one to three sentences on what the section says.
- `obligations`: every obligation in the section. Use an empty list if there are none.

Each `Obligation` has four fields:

- `requirement`: what the firm must do, in plain English.
- `paragraph`: the paragraph number it comes from, for example `"4.2"`.
- `deadline`: the deadline exactly as written in the document, or `null` if there is none.
- `evidence`: a quote copied word for word from that paragraph, at most 30 words.

For example:

```json
{
  "number": "7",
  "title": "Incident reporting",
  "summary": "Firms must report major incidents to the regulator quickly.",
  "obligations": [
    {
      "requirement": "Report major incidents to the regulator",
      "paragraph": "7.1

In [6]:
print(json.dumps(SectionSummary.model_json_schema()["properties"], indent=1)[:900], "...")

{
 "number": {
  "description": "The section number, for example '4'",
  "title": "Number",
  "type": "string"
 },
 "title": {
  "description": "The section title",
  "title": "Title",
  "type": "string"
 },
 "summary": {
  "description": "One to three sentences on what the section says",
  "title": "Summary",
  "type": "string"
 },
 "obligations": {
  "description": "Every obligation in the section; empty if there are none",
  "items": {
   "$ref": "#/$defs/Obligation"
  },
  "title": "Obligations",
  "type": "array"
 }
} ...


Let's try the sub-agent on its own, on the incident reporting section. Here we wrap it in a deep agent of its own,
using the same instructions and response format, so we can see what it returns before the main agent uses it.

In [7]:
from deepagents import create_deep_agent

summariser = create_deep_agent(
    model=get_model("section"),
    system_prompt=SUB_AGENT_INSTRUCTIONS,
    response_format=SectionSummary,
)

incident_reporting = document["sections"][6]
task = f"Section {incident_reporting['number']}: {incident_reporting['title']}\n\n{incident_reporting['text']}"
summary = summariser.invoke({"messages": [{"role": "user", "content": task}]})["structured_response"]

print(summary.summary, "\n")
for obligation in summary.obligations:
    print(f"- [{obligation.paragraph}] {obligation.requirement} (deadline: {obligation.deadline})")

Firms must notify the FSOA of major incidents in three stages: an initial notification, an intermediate report and a final report. Responsibility for reporting stays with the firm when a third-party provider causes the incident. The FSOA will publish a template, and firms may use their existing formats until then. 

- [7.1] Submit an initial notification to the FSOA within 4 hours of classifying an incident as major, and in any case no later than 24 hours after first detection (deadline: within 4 hours of classifying an incident as a major incident, and in any case no later than 24 hours after the incident is first detected)
- [7.2] Submit an intermediate report setting out the cause (as far as known), the services and customers affected, and actions taken so far (deadline: within 72 hours of the initial notification)
- [7.3] Submit a final report after the incident is resolved (deadline: within 30 business days of the incident being resolved)
- [7.3] Include a root-cause analysis and 

## Step 3: Check the obligations with plain code

Models sometimes paraphrase when they should quote, or get a date slightly wrong. In a compliance brief, that's
not acceptable. So ordinary Python can check each obligation:

- Does its quote appear **word for word** in the section?
- Does its deadline appear in the section?

These checks are deterministic, so they're cheap, fast and easy to audit.

In [8]:
import re


def normalise(text: str) -> str:
    """Lowercase, straighten quotes, collapse whitespace and trim punctuation, so formatting can't hide a match."""
    text = text.replace("‘", "'").replace("’", "'").replace("“", '"').replace("”", '"')
    return re.sub(r"\s+", " ", text).strip(" .\"'").lower()


def check_summary(summary: SectionSummary, section: dict) -> list[str]:
    """Return a list of problems with a section summary's obligations. An empty list means every one checks out."""
    section_text = normalise(section["text"])
    issues = []
    for i, obligation in enumerate(summary.obligations, start=1):
        if normalise(obligation.evidence) not in section_text:
            issues.append(f"Obligation {i}: the quote is not word for word from section {section['number']}.")
        elif obligation.deadline and normalise(obligation.deadline) not in section_text:
            issues.append(f"Obligation {i}: the deadline '{obligation.deadline}' is not in section {section['number']}.")
    return issues


print(check_summary(summary, incident_reporting) or "Every obligation checks out.")

Every obligation checks out.


## Step 4: The main agent

The main agent is defined in `agent.py`. It has:

- **Instructions** (`instructions.md`): load the document, give each section to its own `section-summariser`
  sub-agent in parallel, collect the summaries in section order, and combine them into a brief for the firm.
- **Tools**: `get_document` and `get_firm_profile`.
- **The sub-agent**: `section_summariser_subagent`, passed as `subagents`.

A deep agent is a complete harness, including a virtual filesystem and a `task` tool, which is how it calls a sub-agent.
That's what makes the fan-out possible without us writing any graph code.

In [9]:
from agent import INSTRUCTIONS, agent

print(INSTRUCTIONS)

# Document summary agent

You are a compliance analyst summarising a regulatory document, one section at a time.
The document is split at its level-2 (`##`) headings. Each section is passed to a sub-agent, which summarises that section on
its own, so a sub-agent only ever sees one section.

## Orchestrator

Given a document ID:

1. Look up the document and read its title.
2. Split the document into sections. Each section keeps its number, title and full text.
3. Pass each section to its own `section-summariser` sub-agent. Run the sub-agents in parallel.
4. Collect one `SectionSummary` back from each sub-agent, and put them in section order.
5. Combine the section summaries into a brief for the firm. Do not add obligations that no sub-agent found, and do not
   drop any that they did.



## Step 5: Run it

Give the main agent a document and a firm. Then we read back what happened: each `task` call is one run of the
`section-summariser` sub-agent. Notice that the model makes all ten calls in the same step. That's the parallel
fan-out.

In [10]:
result = agent.invoke({"messages": [{"role": "user", "content": "Summarise document PS26-4 for FIRM-001"}]})

for message in result["messages"]:
    for call in getattr(message, "tool_calls", []):
        print("called:", call["name"], call["args"].get("document_id") or call["args"].get("subagent_type") or "")

called: get_document PS26-4
called: get_firm_profile 
called: task section-summariser
called: task section-summariser
called: task section-summariser
called: task section-summariser
called: task section-summariser
called: task section-summariser
called: task section-summariser
called: task section-summariser
called: task section-summariser
called: task section-summariser


Each sub-agent returned a `SectionSummary` as JSON. Here is how many obligations it found in each section, and
whether they pass our check:

In [11]:
summaries = [
    SectionSummary(**json.loads(message.content))
    for message in result["messages"]
    if message.type == "tool" and message.name == "task"
]
sections = {section["number"]: section for section in document["sections"]}

for s in sorted(summaries, key=lambda s: int(s.number)):
    problems = check_summary(s, sections[s.number])
    print(f"{s.number:>2}. {s.title:<55} {len(s.obligations):>2} obligations, {len(problems)} problems")

 1. Introduction and purpose                                 0 obligations, 0 problems
 2. Scope and application                                    2 obligations, 0 problems
 3. Definitions                                              0 obligations, 0 problems
 4. Important business services and impact tolerances        7 obligations, 0 problems
 5. Mapping and scenario testing                             9 obligations, 0 problems
 6. Third-party technology providers                        11 obligations, 0 problems
 7. Incident reporting                                       4 obligations, 0 problems
 8. Governance and accountability                            7 obligations, 0 problems
 9. Transitional arrangements                                5 obligations, 0 problems
10. Next steps and contacts                                  0 obligations, 0 problems


Here's the brief the main agent wrote:

In [12]:
final = result["messages"][-1].text
print(final)

# PS26/4 brief for Harbour Mutual Bank (FIRM-001)

**PS26/4: Operational resilience and third-party technology risk.** The rules take effect on **1 January 2027**. All 10 sections were summarised; sections 1, 3 and 10 set no obligations.

## What it means for Harbour Mutual Bank

- **You are not a small firm.** You have 1.2m customers and £9bn in assets, and exceeding either limit (250,000 customers or £5bn) rules you out. The extra time in 9.1 and 9.2 does not apply, so the **2027 deadlines apply to you**.
- **No senior manager is named yet.** This is your most urgent gap. You must name one by **31 January 2027** and tell the FSOA within 14 days of appointing them. There is no transition period for this (9.4).
- **NimbusCloud is a concentration risk.** Mobile banking and card payments both run on it. That triggers 6.5: you must assess the concentration risk and explain to the board how you would manage NimbusCloud failing. Your mapping (5.2) must show this, and NimbusCloud failure is 

## Done! Where to look next

- **LangSmith:** if you set `LANGSMITH_TRACING=true` in `.env`, every run above is traced. Open it to see the ten
  sub-agent runs side by side, and every prompt the model received.
- **LangGraph Studio:** run `uv run langgraph dev` in the `03_document_summary_agent` folder and chat with the
  agent.
- **Managed Deep Agents:** run `uv run mda dev` to try the `agent` definition as a deep agent, and
  `uv run mda deploy` to deploy it. See the README.

## Next Steps:
1. **Datasets, Evals and Experiments** Setup the infrastructure to be able to experiment with the agent.
2. **Presentation agent.** Separate the main agent so it returns raw data and passes it to a presentation agent for final report generation. Create datasets and experiements to allow for testing of data restrieval separate to presentation.

## Stretch goals
1. **Verify response within agent.** Give the agents a tool or middleware (e.g RubricMiddleware) that verifies the outputs and re-runs any
   sub-agent whose summary has problems.
2. **Human review.** Add human-in-the-loop, so a reviewer must approve the brief before it's final.
3. **Performance tuning.** Do all agents and sub-agents need the same model? Optimize for cost, performance and accuracy.



## What this notebook adds to the starter

The cells above are the starter's. They differ in one place only: where a model is chosen, which now goes through `get_model`. By default `get_model` answers from recorded model calls, so the notebook needs no key and makes no live call. Step 0 describes the starter's own setup with a provider key, which applies only when `MODEL_BACKEND` is set to `langchain` in the `.env` in the repository root.

Following the same document, PS26-4, for the same firm, the sections below add four things:

1. The starter's section summaries, scored against reference labels for every obligation in the document.
2. The same document read into cited facts: each paragraph a unit, each obligation a fact with its word-for-word quote, its paragraph and the results of plain-code checks, and a brief whose statements each cite the facts they rest on.
3. The same scoring of this repository's section 7, beside the starter's.
4. The place where a named person reviews and signs.

The reference labels were made by two independent labelling agents, each reading every paragraph under written rules fixed before either started. The coordinating session settled each disagreement between them under those rules.

## Scoring the starter's agent

Step 3 checked each quote and deadline against its section, but it did not say how many of the document's obligations the agent found. The reference labels list every obligation in PS26-4 with its paragraph, its quote and its deadline as written. The code in `evals/` scores a section summary against them on seven measures. Each measure is a count over a stated population, and none of them calls a model.

Scoring is done by the next cell on the summaries the starter's agent returned in step 5, one example per section. It prints the seven totals with their counts, then the labelled obligations found in each section.

In [13]:
from evals.build_dataset import read_examples
from evals.evaluators import MEASURE_KEYS, evaluate_pair
from fsi_common.evals.runner import run_evaluators

section_examples = read_examples("sections.jsonl")


class ScoringFailedError(Exception):
    """The scoring run could not score what it was given."""


def measure_lines(measures):
    """One line per measure: its counts, then what was counted."""
    return [m.line() + ("  (lower is better)" if m.lower_is_better else "") for m in measures]


def score_sections(summaries_by_number):
    """Score a summary for each section against its labels, and total each measure."""
    pairs = []
    for example in section_examples:
        number = example["inputs"]["section"]["number"]
        summary = summaries_by_number.get(number)
        output = summary.model_dump() if summary else {"number": number, "obligations": []}
        pairs.append((example, output))
    report = run_evaluators(pairs, [evaluate_pair], required=MEASURE_KEYS)
    if report.failures:
        raise ScoringFailedError("; ".join(report.failures))
    return report


starter_by_number = {s.number: s for s in summaries}
starter_report = score_sections(starter_by_number)

print(f"{len(starter_by_number)} of {len(section_examples)} sections returned a summary\n")
print("\n".join(measure_lines(starter_report.totals)))
print()
for name, measures in starter_report.per_example:
    found = next(m for m in measures if m.key == "obligations_found")
    print(f"{name:<18} {found.line()}")

10 of 10 sections returned a summary

obligations_found: 37 of 39 labelled obligations in scope
invented_obligations: 6 of 45 predicted obligations  (lower is better)
quotes_word_for_word: 45 of 45 predicted obligations citing a paragraph in scope
deadlines_as_written: 26 of 26 predicted obligations with a deadline
deadlines_matched: 16 of 16 matched obligations whose labelled deadline is not empty
paragraph_recall: 23 of 23 paragraphs holding a labelled obligation
paragraph_precision: 23 of 28 paragraphs holding a predicted obligation

PS26-4/section/1   obligations_found: none in scope (labelled obligations in scope)
PS26-4/section/2   obligations_found: 1 of 1 labelled obligations in scope
PS26-4/section/3   obligations_found: none in scope (labelled obligations in scope)
PS26-4/section/4   obligations_found: 7 of 7 labelled obligations in scope
PS26-4/section/5   obligations_found: 8 of 8 labelled obligations in scope
PS26-4/section/6   obligations_found: 9 of 10 labelled obligatio

## The same document as cited facts

Now the same document goes through this repository's workflow, for the same firm. The request is limited to section 7, Incident reporting, so that one run shows every step. The code handles every section, and a request without the filter reads all of them.

The next cell starts the run through `BriefRuns`, the service the reviewer's application also uses, and waits until the run stops at the review step. The run ends there on purpose: a named person takes over, and nothing in this notebook approves anything.

In [14]:
import time

from workflow.requests import brief_request_text
from workflow.runs import BriefRuns

BRIEF_REQUEST = brief_request_text("PS26-4", "FIRM-001")
WAIT_LIMIT_SECONDS = 900


class RunDidNotReachReviewError(Exception):
    """The run failed, or ran out of time, before it reached the review step."""


runs = BriefRuns()
thread_id = runs.start(
    {"document_id": "PS26-4", "firm_id": "FIRM-001", "request": BRIEF_REQUEST, "sections": ["7"]}
)

give_up_at = time.monotonic() + WAIT_LIMIT_SECONDS
while runs.status(thread_id) == "running":
    if time.monotonic() > give_up_at:
        raise RunDidNotReachReviewError(f"The run was still going after {WAIT_LIMIT_SECONDS} seconds.")
    time.sleep(0.5)

status = runs.status(thread_id)
if status != "awaiting_review":
    raise RunDidNotReachReviewError(f"The run ended as {status}: {runs.error(thread_id) or 'no failure recorded'}")

state = runs.result(thread_id)
print(f"The run is {status}.")

The run is awaiting_review.


### Every paragraph accounted for

The context layer reads each source into units, one for each paragraph of the document and one for each row of a table. Every unit has to end in one of two places: with at least one fact, or on a list of units that hold none, each with its reason. The run stops when the counts do not close.

Counts for each source the run read are printed by the next cell, and the reason for each unit that holds no fact.

In [15]:
from fsi_common.records import Fact, Source, UnitAccounting
from utils.records import FACT_CONTEXT

for source_id, record in state["accounting"].items():
    accounting = UnitAccounting.model_validate(record)
    source = Source.model_validate(state["sources"][source_id])
    accounted = len(accounting.with_facts) + len(accounting.without_facts)
    print(
        f"{source.title} ({source.key}): {accounted} of {accounting.units_in} units accounted for, "
        f"{len(accounting.with_facts)} with facts and {len(accounting.without_facts)} with none"
    )
    for unit in accounting.without_facts:
        print(f"    {unit.anchor} holds no fact: {unit.reason}")

Third-party providers (third_party_providers): 2 of 2 units accounted for, 2 with facts and 0 with none
Firm profiles (firms): 1 of 1 units accounted for, 1 with facts and 0 with none
Policy Statement PS26/4: Operational resilience and third-party technology risk (PS26-4): 5 of 5 units accounted for, 3 with facts and 2 with none
    para:7.4 holds no fact: The section reader found no obligation in this paragraph.
    para:7.5 holds no fact: The section reader found no obligation in this paragraph.


### Each obligation as a cited fact

Each obligation the section's sub-agent returned becomes a fact. A fact holds its kind, the paragraph it is about, a quote that must be word for word in that paragraph, the deadline as written, and the result of each plain-code check. The brief may cite only a fact that passed every check. A fact that failed stays on the record, marked, and the brief does not use it.

The cell after it counts the facts by kind and by outcome, then lists the obligations with their checks.

In [16]:
from collections import Counter

facts = {
    fact_id: Fact.model_validate(contents[0], context=FACT_CONTEXT)
    for fact_id, contents in state["facts"].items()
}
several_readings = [fact_id for fact_id, contents in state["facts"].items() if len(contents) > 1]
by_kind = Counter(fact.kind for fact in facts.values())

print("Facts by kind: " + ", ".join(f"{n} {kind}" for kind, n in sorted(by_kind.items())))
print(f"Facts that passed every plain-code check: {sum(f.passed_checks for f in facts.values())} of {len(facts)}")
print(f"Facts that hold more than one reading: {len(several_readings)} of {len(facts)}\n")


def paragraph_order(fact):
    """Sort key for an obligation's paragraph anchor, such as para:7.1."""
    return tuple(int(part) for part in fact.subject.anchor.removeprefix("para:").split("."))


for fact in sorted((f for f in facts.values() if f.kind == "obligation"), key=paragraph_order):
    requirements = "; ".join(fact.attributes["requirement"])
    print(f"[{fact.subject.anchor}] {'passed' if fact.passed_checks else 'FAILED'}: {requirements}")
    print(f"    deadline: {fact.attributes.get('deadline') or 'none'}")
    print(f'    quote: "{fact.quote}"')
    for check in fact.checks:
        print(f"    {check.check}: {check.outcome}. {check.detail}")

Facts by kind: 8 firm_attribute, 4 obligation, 4 provider_attribute
Facts that passed every plain-code check: 16 of 16
Facts that hold more than one reading: 0 of 16

[para:7.1] passed: Submit an initial notification to the FSOA within 4 hours of classifying an incident as major, and in any case no later than 24 hours after the incident is first detected
    deadline: within 4 hours of classifying an incident as a major incident, and in any case no later than 24 hours after the incident is first detected
    quote: "Firms must submit an initial notification to the FSOA within 4 hours of classifying an incident as a major incident"
    quote-verbatim.v1: pass. found word for word in paragraph 7.1
    deadline-verbatim.v1: pass. found word for word in paragraph 7.1
[para:7.2] passed: Submit an intermediate report setting out the cause of the incident as far as known, the services and customers affected, and the actions taken so far
    deadline: within 72 hours of the initial notificatio

### The brief and the facts each statement cites

The brief is written from the facts that passed. Each statement names the facts it rests on, and the review request carries each cited fact with the full text of its paragraph, so a reviewer can read the source beside the claim. Questions the sources do not settle go to the reviewer as open questions, each addressed to a role.

Each statement is printed by the next cell with the paragraph and quote of every fact it cites, then the open questions.

In [17]:
review = runs.review_request(thread_id)

cited = [cited_fact for statement in review.statements for cited_fact in statement.facts]
distinct = {cited_fact.fact.id for cited_fact in cited}
passed = {cited_fact.fact.id for cited_fact in cited if cited_fact.fact.passed_checks}
print(f"Statements in the brief: {len(review.statements)}")
print(f"Distinct facts cited: {len(distinct)}, of which {len(passed)} of {len(distinct)} passed every plain-code check")
print(f"Open questions for the reviewer: {len(review.open_questions)}\n")

for number, statement in enumerate(review.statements, start=1):
    print(f"{number}. {statement.heading or 'No heading'}")
    print(f"   {statement.text}")
    for cited_fact in statement.facts:
        print(f'   cites [{cited_fact.fact.subject.anchor}] "{cited_fact.fact.quote}"')

for question in review.open_questions:
    print(f"Open question about {question.about.anchor} for {question.ask_role}: {question.question}")

Statements in the brief: 9
Distinct facts cited: 16, of which 16 of 16 passed every plain-code check
Open questions for the reviewer: 0

1. About the firm
   FIRM-001 is Harbour Mutual Bank, a retail bank with 1,200,000 customers and total assets of £9 billion.
   cites [row:FIRM-001] ""name": "Harbour Mutual Bank""
   cites [row:FIRM-001] ""type": "retail bank""
   cites [row:FIRM-001] ""customers": 1200000"
   cites [row:FIRM-001] ""total_assets": "£9 billion""
2. About the firm
   No operational resilience senior manager is recorded for the firm.
   cites [row:FIRM-001] ""operational_resilience_senior_manager": null"
3. Third-party providers
   The firm has a material arrangement with NimbusCloud (PROVIDER-001), which provides core banking hosting, under a contract ending on 30 June 2027.
   cites [row:FIRM-001] "{ "provider_id": "PROVIDER-001", "contract_end": "30 June 2027", "material": true }"
   cites [row:PROVIDER-001] ""name": "NimbusCloud""
   cites [row:PROVIDER-001] ""servi

## Scoring this repository's engine on section 7

The measures that scored the starter's agent now score this repository's section sub-agent, on the one section the run covered. The next cell takes the sub-agent's summary of section 7 as the model returned it, in the starter's own `SectionSummary` shape, before any check, and scores it beside the starter's summary of the same section from step 5. The same code scores both against the same labels.

In [18]:
from evals.evaluators import score_summary
from sub_agent import SectionSummary

SECTION = "7"
section_example = next(e for e in section_examples if e["inputs"]["section"]["number"] == SECTION)
ours = SectionSummary.model_validate(state["raw_summaries"][SECTION])

scored = {
    "starter's agent": starter_by_number[SECTION],
    "this repository's engine": ours,
}
by_name = {
    name: {m.key: m for m in score_summary(summary.model_dump(), section_example["reference_outputs"])}
    for name, summary in scored.items()
}

for key in MEASURE_KEYS:
    print(key)
    for name, measures in by_name.items():
        print(f"    {name:<26} {measures[key].line().removeprefix(key + ': ')}")

obligations_found
    starter's agent            4 of 4 labelled obligations in scope
    this repository's engine   4 of 4 labelled obligations in scope
invented_obligations
    starter's agent            0 of 4 predicted obligations
    this repository's engine   0 of 4 predicted obligations
quotes_word_for_word
    starter's agent            4 of 4 predicted obligations citing a paragraph in scope
    this repository's engine   4 of 4 predicted obligations citing a paragraph in scope
deadlines_as_written
    starter's agent            3 of 3 predicted obligations with a deadline
    this repository's engine   3 of 3 predicted obligations with a deadline
deadlines_matched
    starter's agent            3 of 3 matched obligations whose labelled deadline is not empty
    this repository's engine   3 of 3 matched obligations whose labelled deadline is not empty
paragraph_recall
    starter's agent            3 of 3 paragraphs holding a labelled obligation
    this repository's engine   

## Where a person signs

The run above stopped at the review step. It built no decision and no sign-off, because the workflow never decides for a person. In the reviewer's application a named person opens each statement, reads the source paragraph beside it, accepts, amends or rejects each fact in their own words, and then signs the brief or returns it. The application runs this same workflow through `BriefRuns` and takes the review from there.

To start the application, run `make serve` from the repository root. It listens on the host and port set by `APP_HOST` and `APP_PORT`. Sign in with your name and role.